In [4]:
import pandas as pd
import numpy as np

In [5]:
df_ratings = pd.read_csv('../data/ratings.csv')
df_movies = pd.read_csv('../data/df_final.csv')

### preping data

In [6]:
df_movies.head()

,Unnamed: 0,movieId,title,genres,budget,revenue,runtime,release_date,vote_average_tmdb,vote_count_tmdb,mean_rating,num_votes,std_rating,all_tags
0,0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,30000000.0,401157969.0,81.0,1995-11-22,7.971,19514.0,3.920930,215.0,0.834859,pixar pixar fun
1,1,2,Jumanji (1995),Adventure|Children|Fantasy,65000000.0,262821940.0,104.0,1995-12-15,7.244,11078.0,3.431818,110.0,0.881713,fantasy magic board game Robin Williams game
2,2,3,Grumpier Old Men (1995),Comedy|Romance,25000000.0,71500000.0,101.0,1995-12-22,6.482,415.0,3.259615,52.0,1.054823,moldy old
3,3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,16000000.0,81452156.0,123.0,1995-12-22,6.234,190.0,2.357143,7.0,0.852168,NaN
4,4,5,Father of the Bride Part II (1995),Comedy,0.0,76594107.0,106.0,1995-12-08,6.300,788.0,3.071429,49.0,0.907148,pregnancy remake


In [7]:
df_movies_prep = df_movies[["genres", "budget", "revenue", "runtime", "vote_count_tmdb", "vote_average_tmdb"]]

In [8]:
print(df_movies_prep.isna().sum())

genres                 0
budget               121
revenue              121
runtime              121
vote_count_tmdb      121
vote_average_tmdb    121
dtype: int64


In [9]:
df_movies_prep['budget'] = df_movies_prep['budget'].fillna(df_movies_prep['budget'].median())
df_movies_prep['revenue'] = df_movies_prep['revenue'].fillna(df_movies_prep['revenue'].median())
df_movies_prep['runtime'] = df_movies_prep['runtime'].fillna(df_movies_prep['runtime'].mean())
df_movies_prep['vote_count_tmdb'] = df_movies_prep['vote_count_tmdb'].fillna(df_movies_prep['vote_count_tmdb'].median())
df_movies_prep['vote_average_tmdb'] = df_movies_prep['vote_average_tmdb'].fillna(df_movies_prep['vote_average_tmdb'].mean())

In [10]:
df_movies_prep['genres'] = df_movies_prep['genres'].apply(lambda x: x.split('|'))
genres = set(g for G in df_movies_prep['genres'] for g in G)

for g in genres:
    df_movies_prep[g] = df_movies_prep.genres.transform(lambda x : 1.5 if(g in x) else 0)

df_movies_prep = df_movies_prep.drop(columns=['genres'])

In [11]:
from sklearn.preprocessing import MinMaxScaler

df_movies_prep['budget'] = np.log1p(df_movies_prep['budget'])
df_movies_prep['revenue'] = np.log1p(df_movies_prep['revenue'])
df_movies_prep['vote_count_tmdb'] = np.log1p(df_movies_prep['vote_count_tmdb'])

num_cols = ['budget', 'revenue', 'runtime', 'vote_average_tmdb', 'vote_count_tmdb']

scaler = MinMaxScaler()
df_movies_prep[num_cols] = scaler.fit_transform(df_movies_prep[num_cols])


In [12]:
df_movies_prep

,budget,revenue,runtime,vote_count_tmdb,vote_average_tmdb,Western,Adventure,Film-Noir,Comedy,Musical,...,Thriller,Sci-Fi,Action,Drama,Romance,IMAX,(no genres listed),Horror,War,Animation
0,0.871598,0.908871,0.138937,0.935007,0.895316,0.0,1.5,0.0,1.5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.5
1,0.910740,0.889470,0.178388,0.881424,0.813658,0.0,1.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.862368,0.829744,0.173242,0.570783,0.728069,0.0,0.0,0.0,1.5,0.0,...,0.0,0.0,0.0,0.0,1.5,0.0,0.0,0.0,0.0,0.0
3,0.839774,0.835723,0.210978,0.497109,0.700213,0.0,0.0,0.0,1.5,0.0,...,0.0,0.0,0.0,1.5,1.5,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.832902,0.181818,0.631365,0.707627,0.0,0.0,0.0,1.5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9737,0.000000,0.603061,0.171527,0.434912,0.842413,0.0,0.0,0.0,1.5,0.0,...,0.0,0.0,1.5,0.0,0.0,0.0,0.0,0.0,0.0,1.5
9738,0.000000,0.718704,0.181818,0.573253,0.878019,0.0,0.0,0.0,1.5,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.5
9739,0.000000,0.000000,0.164666,0.278681,0.763787,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.5,0.0,0.0,0.0,0.0,0.0,0.0
9740,0.000000,0.673503,0.154374,0.493059,0.910929,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.5,0.0,0.0,0.0,0.0,0.0,0.0,1.5


### cosine similariry

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(df_movies_prep, df_movies_prep)
print(f"Dimensions of our genres cosine similarity matrix: {cosine_sim.shape}")

Dimensions of our genres cosine similarity matrix: (9742, 9742)


In [14]:
from fuzzywuzzy import process

def movie_finder(title):
    all_titles = df_movies['title'].tolist()
    closest_match = process.extractOne(title, all_titles)
    return closest_match[0]

c:\Users\Kelly\akelly\ENSEA\3A\MASTER\uec10_bigdata\poubs\Big_Data_Project\.venv\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [15]:
def because_you_watched(movie_title : str, n_recommandations : int):
    # au cas ou, y'a des erreurs, on utilise le movie finder
    title = movie_finder(movie_title)

    # on cherche l'index
    movie_idx = dict(zip(df_movies['title'], list(df_movies.index)))
    idx = movie_idx[title]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:(n_recommandations+1)]
    similar_movies = [i[0] for i in sim_scores]

    print(f"Because you watched {title}:")
    print(df_movies['title'].iloc[similar_movies])
    

In [16]:
because_you_watched("evangelion", 10)

Because you watched Evangelion: 1.0 You Are (Not) Alone (Evangerion shin gekijôban: Jo) (2007):
8032       Batman: The Dark Knight Returns, Part 1 (2012)
8841                         Patlabor 2: The Movie (1993)
3546                          Spriggan (Supurigan) (1998)
8246          Justice League: Crisis on Two Earths (2010)
8048                                       Redline (2009)
8940                        Ghost in the Shell 2.0 (2008)
9488                           Ultimate Avengers 2 (2006)
8792    Ghost in the Shell Arise - Border 1: Ghost Pai...
9547                    Final Flight of the Osiris (2003)
973                                          Akira (1988)
Name: title, dtype: str
